^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl.metadata (52 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 985.5 kB/s eta 0:00:01
   --------------- ------------------------ 0.5/1.4 MB 985.5 kB/s eta 0:00:01
   --------------- ------------------------ 0.5/1.4 MB 985.5 kB/s eta 0:00:01
   --------------- ------------------------ 0.5/1.4 MB 985.5 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 441.3 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.4 MB 441.3 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.4 MB 441.3 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.4 MB 441.3 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.4 MB 441.3 kB/s eta 0:00:02
   ------------

In [3]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt
import networkx as nx

# Part A: Data Preparation
print("="*60)
print("MARKET BASKET ANALYSIS")
print("="*60)

# Sample transaction data
transactions = [
    ['Bread', 'Milk', 'Eggs'],
    ['Bread', 'Butter', 'Jam'],
    ['Milk', 'Bread', 'Butter'],
    ['Bread', 'Milk', 'Butter', 'Eggs'],
    ['Milk', 'Eggs', 'Cereal'],
    ['Bread', 'Eggs', 'Milk'],
    ['Butter', 'Jam', 'Bread'],
    ['Bread', 'Milk', 'Butter'],
    ['Milk', 'Cereal', 'Eggs'],
    ['Bread', 'Butter', 'Jam', 'Milk']
]

print(f"\nTotal transactions: {len(transactions)}")
print("\nSample transactions:")
for i in range(3):
    print(f"Transaction {i+1}: {transactions[i]}")

# Create binary matrix
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("\nBinary matrix (first 5 rows):")
print(df.head())

# Basic statistics
print("\n" + "="*60)
print("BASIC STATISTICS")
print("="*60)

# Item frequencies
item_freq = df.sum().sort_values(ascending=False)
print("\nItem frequencies:")
print(item_freq)

# Transaction lengths
trans_lengths = [len(t) for t in transactions]
print(f"\nAverage items per transaction: {np.mean(trans_lengths):.2f}")
print(f"Min items: {min(trans_lengths)}")
print(f"Max items: {max(trans_lengths)}")

MARKET BASKET ANALYSIS

Total transactions: 10

Sample transactions:
Transaction 1: ['Bread', 'Milk', 'Eggs']
Transaction 2: ['Bread', 'Butter', 'Jam']
Transaction 3: ['Milk', 'Bread', 'Butter']

Binary matrix (first 5 rows):
   Bread  Butter  Cereal   Eggs    Jam   Milk
0   True   False   False   True  False   True
1   True    True   False  False   True  False
2   True    True   False  False  False   True
3   True    True   False   True  False   True
4  False   False    True   True  False   True

BASIC STATISTICS

Item frequencies:
Bread     8
Milk      8
Butter    6
Eggs      5
Jam       3
Cereal    2
dtype: int64

Average items per transaction: 3.20
Min items: 3
Max items: 4


In [ ]:
# Part B: Apriori Algorithm
print("\n" + "="*60)
print("APRIORI RESULTS")
print("="*60)

# Find frequent itemsets
min_support = 0.3 # 30% of transactions
frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)

print(f"\nFrequent itemsets (support >= {min_support}):")
print(f"Total: {len(frequent_itemsets)}")
print(frequent_itemsets)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print(f"\nAssociation rules generated: {len(rules)}")

# Filter rules
rules = rules[rules['confidence'] >= 0.5]
print(f"\nRules with confidence >= 0.5: {len(rules)}")

print("\nAll rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

In [ ]:
# Part C: Top Rules
print("\n" + "="*60)
print("TOP RULES")
print("="*60)

# Top 5 by lift
print("\nTOP 5 RULES BY LIFT:")
top_lift = rules.nlargest(5, 'lift')
for i, (idx, row) in enumerate(top_lift.iterrows(), 1):
    ante = ', '.join(list(row['antecedents']))
    cons = ', '.join(list(row['consequents']))
    print(f"{i}. {ante} -> {cons}")
    print(f" Support: {row['support']:.2f}, Confidence: {row['confidence']:.2f}, Lift: {row['lift']:.2f}")

# Top 5 by confidence
print("\nTOP 5 RULES BY CONFIDENCE:")
top_conf = rules.nlargest(5, 'confidence')
for i, (idx, row) in enumerate(top_conf.iterrows(), 1):
    ante = ', '.join(list(row['antecedents']))
    cons = ', '.join(list(row['consequents']))
    print(f"{i}. {ante} -> {cons}")
    print(f" Support: {row['support']:.2f}, Confidence: {row['confidence']:.2f}, Lift: {row['lift']:.2f}")

# Simple visualization
plt.figure(figsize=(10, 6))
plt.bar(range(len(item_freq)), item_freq.values)
plt.xticks(range(len(item_freq)), item_freq.index, rotation=45)
plt.title('Item Frequencies')
plt.xlabel('Items')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Simple network graph
if len(rules) > 0:
    plt.figure(figsize=(8, 6))
    G = nx.DiGraph()
    
    # Add top 5 rules to graph
    for idx, row in top_lift.iterrows():
        ante = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        G.add_edge(ante, cons, weight=row['lift'])
        
    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', 
            node_size=2000, font_size=10, font_weight='bold',
            arrows=True, arrowstyle='->', arrowsize=20)
    plt.title('Top Rules Network')
    plt.show()

In [ ]:
# Business Interpretation
print("\n" + "="*60)
print("BUSINESS INTERPRETATION")
print("="*60)

print("\nWhat the rules mean:")
print(" - If a customer buys [item A], they are likely to also buy [item B]")
print(" - Lift > 1 means items are positively associated")
print(" - Confidence shows how often the rule holds true")

print("\nKey Findings:")
if len(rules) > 0:
    best_rule = rules.nlargest(1, 'lift').iloc[0]
    ante = ', '.join(list(best_rule['antecedents']))
    cons = ', '.join(list(best_rule['consequents']))
    print(f" Strongest association: {ante} -> {cons}")
    print(f" (Lift: {best_rule['lift']:.2f})")

print("\nMarketing Suggestions:")
print("1. Store Layout: Place associated items near each other")
print("2. Promotions: Offer discounts on pairs that are frequently bought together")
print("3. Cross-selling: Train staff to suggest complementary items")
print("4. Product placement: Create themed displays (e.g., breakfast section)")

# Save results
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_csv('simple_rules.csv', index=False)
print("\nResults saved to 'simple_rules.csv'")

# POST LAB

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import seaborn as sns

# POST-LAB: FP-Growth, Different Support Thresholds, and Dashboard

# 1. Different Domain: Mock E-commerce Tech Transactions
tech_transactions = [
    ['Laptop', 'Mouse', 'Keyboard'],
    ['Monitor', 'Keyboard'],
    ['Laptop', 'Mouse', 'Headphones'],
    ['Laptop', 'Monitor', 'Keyboard'],
    ['Smartphone', 'Headphones', 'Charger'],
    ['Laptop', 'Mouse', 'Monitor', 'Keyboard'],
    ['Smartphone', 'Charger', 'Case'],
    ['Tablet', 'Case', 'Charger'],
    ['Laptop', 'Mouse', 'Charger'],
    ['Monitor', 'Headphones', 'Keyboard']
]

te = TransactionEncoder()
te_ary = te.fit(tech_transactions).transform(tech_transactions)
df_tech = pd.DataFrame(te_ary, columns=te.columns_)

# 2. FP-Growth & Analyze Effect of Different Support Thresholds
support_thresholds = [0.1, 0.2, 0.3, 0.4]
itemset_counts = []

for min_sup in support_thresholds:
    freq_items = fpgrowth(df_tech, min_support=min_sup, use_colnames=True)
    itemset_counts.append(len(freq_items))

# Generate Rules with Pruning (lift > 1.2)
final_freq_items = fpgrowth(df_tech, min_support=0.2, use_colnames=True)
rules = association_rules(final_freq_items, metric="lift", min_threshold=1.2)

# 3. Visualization Dashboard
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Support Threshold Analysis
axes[0].plot(support_thresholds, itemset_counts, marker='o', linestyle='-', color='b')
axes[0].set_title('Effect of Min Support on FP-Growth Itemsets')
axes[0].set_xlabel('Minimum Support Threshold')
axes[0].set_ylabel('Number of Frequent Itemsets')
axes[0].grid(True)

# Subplot 2: Rules Scatter Plot (Support vs Confidence colored by Lift)
scatter = axes[1].scatter(rules['support'], rules['confidence'], 
                          c=rules['lift'], cmap='viridis', s=100, alpha=0.7)
axes[1].set_title('Rules Pruning Dashboard (FP-Growth)')
axes[1].set_xlabel('Support')
axes[1].set_ylabel('Confidence')
fig.colorbar(scatter, ax=axes[1], label='Lift')

plt.tight_layout()
plt.show()

print("\nTop 3 Tech Rules (Pruned by Lift):")
top_rules = rules.nlargest(3, 'lift')
for idx, row in top_rules.iterrows():
    print(f"{list(row['antecedents'])} -> {list(row['consequents'])} (Lift: {row['lift']:.2f})")